# Unit 5 — Searching & Complete Search

A contest asks: is target `X` among `N` sorted values, fast — and can you answer many such queries?
This unit compares a direct scan with binary search, then uses complete search when the candidate space is small enough.

## Lesson 1 — Start with Linear Search

Linear search checks values from left to right until it finds the target.
One query takes O(n) time.
That is a useful baseline, but repeating it for many queries can be too slow.

In [ ]:
def solve(data: str) -> str:
    tokens = data.split()
    n = int(tokens[0])
    target = int(tokens[n + 1])
    for position in range(n):
        value = int(tokens[position + 1])
        if value == target:
            return "YES"
    return "NO"

assert solve("6 4 8 15 16 23 42 42") == "YES"

## Binary Search Narrows the Possibilities

Binary search begins by sorting the values.
The variables `lo` and `hi` mark an inclusive range where the target could still be.
This is the loop invariant: before every check, if the target exists in the list, it is somewhere from `lo` through `hi`.
After checking `mid`, discard the half that cannot contain the target.
The remaining range halves each time, so one query takes O(log n) after the O(n log n) sort.

In [ ]:
def solve(data: str) -> str:
    tokens = data.split()
    n = int(tokens[0])
    values = []
    for position in range(n):
        values.append(int(tokens[position + 1]))
    target = int(tokens[n + 1])
    ordered = sorted(values)

    lo = 0
    hi = n - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if ordered[mid] == target:
            return "YES"
        if ordered[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return "NO"

assert solve("7 18 3 11 25 7 14 30 30") == "YES"

## Lower Bounds Turn Search into Counting

A lower bound is the first position whose value is not less than a target.
Here the search range is half-open: `[lo, hi)`, so `hi` itself is not included.
The lower bound of `target + 1` is the first position after every copy of `target`.
Subtracting the two boundary positions counts the copies without scanning them.

In [ ]:
def solve(data: str) -> str:
    tokens = data.split()
    n = int(tokens[0])
    values = []
    for position in range(n):
        values.append(int(tokens[position + 1]))
    target = int(tokens[n + 1])
    ordered = sorted(values)

    def lower_bound(wanted):
        lo = 0
        hi = n
        while lo < hi:
            mid = (lo + hi) // 2
            if ordered[mid] < wanted:
                lo = mid + 1
            else:
                hi = mid
        return lo

    first = lower_bound(target)
    after = lower_bound(target + 1)
    return str(after - first)

assert solve("8 9 2 9 5 1 9 7 9 9") == "4"

## Lesson 2 — Complete Search Tries Every Allowed Choice

When `n` is small, complete search can enumerate every pair with two fixed-depth nested loops.
Starting the second position at `first + 1` uses two different items and visits each unordered pair once.
There are O(n²) pairs, connecting this loop shape to the cost rules from Unit 3.

In [ ]:
def solve(data: str) -> str:
    tokens = data.split()
    n = int(tokens[0])
    target = int(tokens[1])
    values = []
    for position in range(n):
        values.append(int(tokens[position + 2]))

    for first in range(n):
        for second in range(first + 1, n):
            if values[first] + values[second] == target:
                return "YES"
    return "NO"

assert solve("6 27 2 5 9 12 18 25") == "YES"

Three fixed-depth loops enumerate every triple, which costs O(n³).
The same idea can enumerate a small fixed range of candidate values.
Before choosing complete search, estimate the number of loop visits and check that the constraints keep it manageable.

In [ ]:
def solve(data: str) -> str:
    tokens = data.split()
    n = int(tokens[0])
    target = int(tokens[1])
    values = []
    for position in range(n):
        values.append(int(tokens[position + 2]))

    for first in range(n):
        for second in range(first + 1, n):
            for third in range(second + 1, n):
                total = values[first] + values[second] + values[third]
                if total == target:
                    return "YES"
    return "NO"

assert solve("7 59 2 5 9 14 20 31 40") == "YES"

## Lesson 3 — Search Over the Answer

Sometimes we do not search for a value stored in a list.
Instead, we search for the smallest answer that is feasible.
For package capacity, if one capacity works, every larger capacity also works.
That false-then-true pattern is monotone, so binary search can find the first feasible capacity.

In [ ]:
def solve(data: str) -> str:
    tokens = data.split()
    n = int(tokens[0])
    day_limit = int(tokens[1])
    weights = []
    for position in range(n):
        weights.append(int(tokens[position + 2]))

    lo = max(weights)
    hi = sum(weights)
    while lo < hi:
        mid = (lo + hi) // 2
        days = 1
        load = 0
        for weight in weights:
            if load + weight > mid:
                days = days + 1
                load = 0
            load = load + weight
        if days <= day_limit:
            hi = mid
        else:
            lo = mid + 1
    return str(lo)

assert solve("5 3 4 2 7 3 9") == "10"

## NOTE — Techniques Saved for Later

This unit does **not** generate subsets or permutations; that needs recursion and is introduced in Unit 9.
This unit also does **not** use a converging two-pointer scan, where positions at both ends of sorted data walk inward; that technique is introduced in Unit 14.
For pair problems here, use fixed-depth nested loops or sort and binary-search for each complement.

## Submit the Solver

After `solve(data)` works, a contest submission can use this wrapper.
It is marked `no-exec` because notebook execution has no contest input waiting for it.

In [ ]:
import sys
print(solve(sys.stdin.read()))